In [1]:
import mne
import numpy as np
from pathlib import Path 
from scipy.io import loadmat
from mne.preprocessing import EOGRegression
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import cohen_kappa_score

In [2]:
np.random.seed(0)

In [3]:
data_folder = Path(r'C:\Users\admin\BCI project\data')
subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
ch_renamed = {
    'EEG-Fz':'Fz',
    'EEG-0':'FC3',
    'EEG-1':'FC1',
    'EEG-2':'FCz',
    'EEG-3':'FC2',
    'EEG-4':'FC4',
    'EEG-5':'C5',
    'EEG-C3':'C3',
    'EEG-6':'C1',
    'EEG-Cz':'Cz',
    'EEG-7':'C2',
    'EEG-C4':'C4',
    'EEG-8':'C6',
    'EEG-9':'CP3',
    'EEG-10':'CP1',
    'EEG-11':'CPz',
    'EEG-12':'CP2',
    'EEG-13':'CP4',
    'EEG-14':'P1',
    'EEG-Pz':'Pz',
    'EEG-15':'P2',
    'EEG-16':'POz',
}
motor_tasks = {1:'left hand vs rest', 2:'right hand vs rest', 3:'feet vs rest', 4:'tongue vs rest'}

In [4]:
def load_test_labels(subject, motor_tasks):
    labels_test_dict = {}
    mat_files = loadmat(Path(r'C:\Users\admin\BCI project\data\true labels') / f'{subject}E.mat')
    labels_test = mat_files['classlabel'].flatten()
    for task in motor_tasks:
        labels_binary = (labels_test == task).astype(int)
        labels_test_dict[task] = labels_binary
    return labels_test_dict, labels_test

In [5]:
def preprocessing(raw, ch_names):
    raw.rename_channels(ch_names)
    raw.set_channel_types({
        'EOG-left':'eog',
        'EOG-central':'eog',
        'EOG-right':'eog'
    })
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing = 'ignore')
    raw.filter(l_freq = 1, h_freq = None, fir_design='firwin')
    raw.set_eeg_reference()
    return raw

In [6]:
def calibration_eog(raw, subject):
    if subject == 'A04':
        calibration = raw.copy().crop(0, 60)
    else:
        calibration = raw.copy().crop(0, 300)
    return calibration

In [7]:
def regression_eog(raw, calibration):
    model_plain = EOGRegression(picks='eeg', picks_artifact='eog').fit(calibration)
    raw_clean_plain = model_plain.apply(raw)
    return raw_clean_plain

In [8]:
def filter_bank(raw):
    raw_dict = {}
    frequency_bands = [(4,8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
    for l_freq, h_freq in frequency_bands:
        raw_dict[(l_freq, h_freq)] = raw.copy().filter(l_freq, h_freq, fir_design='firwin')
    return raw_dict

In [9]:
def epoching_train(raw_dict, motor_tasks):
    epochs_dict = {}
    epochs_cropped_dict = {}
    epochs_data_dict = {}
    epochs_cropped_data_dict = {}
    labels_dict = {}
    
    for band in raw_dict:    
        events, event_id = mne.events_from_annotations(raw_dict[band])
        classes = {'left hand': event_id['769'], 'right hand': event_id['770'], 
                 'feet': event_id['771'], 'tongue': event_id['772']}
        epochs =  mne.Epochs(raw_dict[band], events, event_id = classes, tmin = -1, 
                             tmax = 4, picks = 'eeg', baseline = None, preload = True)
        epochs_cropped = epochs.copy().crop(tmin = 1.0, tmax = 2)
        epochs_data = epochs.get_data(copy = False)
        epochs_cropped_data = epochs_cropped.get_data(copy = False)
        
        labels = epochs.events[:, -1] - event_id['769'] + 1
        for task in motor_tasks:
            labels_binary = (labels == task).astype(int)
            labels_dict[task] = labels_binary
    
        epochs_dict[band] = epochs
        epochs_cropped_dict[band] = epochs_cropped
        epochs_data_dict[band] = epochs_data
        epochs_cropped_data_dict[band] = epochs_cropped_data
    
    return epochs_dict, epochs_cropped_dict, epochs_data_dict, epochs_cropped_data_dict, labels_dict

In [10]:
def epoching_test(raw_dict):
    epochs_dict = {}
    epochs_cropped_dict = {}
    epochs_data_dict = {}
    epochs_cropped_data_dict = {}

    for band in raw_dict:
        
        events, event_id = mne.events_from_annotations(raw_dict[band])
        epochs = mne.Epochs(raw_dict[band], events, event_id = {'unknown':event_id['783']}, tmin = -1,
                            tmax = 4, picks = 'eeg', baseline = None, preload = True)
        epochs_cropped = epochs.copy().crop(tmin = 1.0, tmax = 2)
        epochs_data = epochs.get_data(copy = False)
        epochs_cropped_data = epochs_cropped.get_data(copy = False)

        epochs_dict[band] = epochs
        epochs_cropped_dict[band] = epochs_cropped
        epochs_data_dict[band] = epochs_data
        epochs_cropped_data_dict[band] = epochs_cropped_data
        
    return epochs_dict, epochs_cropped_dict, epochs_data_dict, epochs_cropped_data_dict

In [11]:
results = {}
class_balance = {1:[], 2:[], 3:[], 4:[]}
accuracy_dict = {}
true_labels_dict = {}
predicted_classes_dict = {}

for subject in subjects:
    
    train_files = data_folder / f'{subject}T.gdf' 
    test_files = data_folder / f'{subject}E.gdf'
    raw_train = mne.io.read_raw_gdf(train_files, preload = True)
    raw_test = mne.io.read_raw_gdf(test_files,  preload = True)

    labels_test_dict, true_labels = load_test_labels(subject, motor_tasks)

    raw_train = preprocessing(raw_train, ch_renamed)
    raw_test = preprocessing(raw_test, ch_renamed)

    calibration_train = calibration_eog(raw_train, subject)
    calibration_test = calibration_eog(raw_test, subject)

    raw_train = regression_eog(raw_train, calibration_train)
    raw_test = regression_eog(raw_test, calibration_test)

    raw_train_dict = filter_bank(raw_train)
    raw_test_dict = filter_bank(raw_test)

    (epochs_train_dict, epochs_train_cropped_dict, epochs_train_data_dict,
     epochs_train_cropped_data_dict, labels_train_dict) = epoching_train(raw_train_dict, motor_tasks)

    (epochs_test_dict, epochs_test_cropped_dict, epochs_test_data_dict,
     epochs_test_cropped_data_dict) = epoching_test(raw_test_dict)
    
    class_components = {}
    class_scores = {}
    results[subject] = {}
    
    for task in motor_tasks:
        X_train_list = []
        X_test_list = []
        csp_dict = {}
        
        lda = LinearDiscriminantAnalysis()
        selector = SelectKBest(score_func=mutual_info_classif, k = 6)
        
        for band in epochs_train_cropped_data_dict:
            csp = CSP(n_components = 4, reg='ledoit_wolf', log = True, norm_trace = False)
            csp_dict[band] = csp
            X_train = csp.fit_transform(epochs_train_cropped_data_dict[band], labels_train_dict[task])
            X_train_list.append(X_train)
        X_train_concatenated = np.concatenate(X_train_list, axis = 1)
        X_train_selected = selector.fit_transform(X_train_concatenated, labels_train_dict[task])
        
        for band in epochs_test_cropped_data_dict:
            X_test = csp_dict[band].transform(epochs_test_cropped_data_dict[band])
            X_test_list.append(X_test)
        X_test_concatenated = np.concatenate(X_test_list, axis = 1)
        X_test_selected = selector.transform(X_test_concatenated)
        
        lda.fit(X_train_selected, labels_train_dict[task])
        class_scores[task] = lda.decision_function(X_test_selected)

        class_components[task] = {'csp_dict':csp_dict, 'selector':selector, 'lda':lda}
        
        score = lda.score(X_test_selected, labels_test_dict[task])
        results[subject][task] = score
        class_balance[task].append(np.mean(labels_test_dict[task] == labels_test_dict[task][0]))

    scores_matrix = np.column_stack([class_scores[1], class_scores[2], class_scores[3], class_scores[4]])
    predicted_classes = np.argmax(scores_matrix, axis=1) + 1
    accuracy = np.mean(predicted_classes == true_labels)

    accuracy_dict[subject] = accuracy                    
    true_labels_dict[subject] = true_labels              
    predicted_classes_dict[subject] = predicted_classes   

Extracting GDF parameters from C:\Users\admin\BCI project\data\A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...


C:\Users\admin\.conda\envs\phd_bci\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Reading 0 ... 672527  =      0.000 ...  2690.108 secs...
Extracting GDF parameters from C:\Users\admin\BCI project\data\A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 686999  =      0.000 ...  2747.996 secs...


C:\Users\admin\.conda\envs\phd_bci\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.3

...

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.3

C:\Users\admin\.conda\envs\phd_bci\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Extracting GDF parameters from C:\Users\admin\BCI project\data\A09E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 675097  =      0.000 ...  2700.388 secs...


C:\Users\admin\.conda\envs\phd_bci\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.3

In [12]:
print('=' * 100 )   
for task in motor_tasks:
    task_scores = []
    for subject in subjects:
        task_scores.append(results[subject][task])
    final_score = np.mean(task_scores)
    final_std = np.std(task_scores)
    class_balance_mean = np.mean(class_balance[task])
    chance_level = max(class_balance_mean, 1 - class_balance_mean)
    
    print('=' * 100 )
    print(f'Total classification precision for {motor_tasks[task]} is {final_score * 100}%')
    print(f'Standard deviation for {motor_tasks[task]} is {final_std}')
    print(f'Chance level for {motor_tasks[task]} is {chance_level * 100}%')
    for sbjct in results.keys():
        print(f'subject {sbjct} - {motor_tasks[task]} score {results[sbjct][task]}')
    print('=' * 100 )
print('=' * 100 )

Total classification precision for left hand vs rest is 82.29166666666666%
Standard deviation for left hand vs rest is 0.06744815872376606
Chance level for left hand vs rest is 69.44444444444444%
subject A01 - left hand vs rest score 0.8506944444444444
subject A02 - left hand vs rest score 0.7291666666666666
subject A03 - left hand vs rest score 0.8923611111111112
subject A04 - left hand vs rest score 0.78125
subject A05 - left hand vs rest score 0.7604166666666666
subject A06 - left hand vs rest score 0.75
subject A07 - left hand vs rest score 0.8715277777777778
subject A08 - left hand vs rest score 0.8333333333333334
subject A09 - left hand vs rest score 0.9375
Total classification precision for right hand vs rest is 79.16666666666667%
Standard deviation for right hand vs rest is 0.0746684773902283
Chance level for right hand vs rest is 75.0%
subject A01 - right hand vs rest score 0.8645833333333334
subject A02 - right hand vs rest score 0.7673611111111112
subject A03 - right hand vs

In [13]:
kappa_dict = {}
for subject in subjects:
    kappa_dict[subject] = cohen_kappa_score(true_labels_dict[subject], predicted_classes_dict[subject])

final_kappa = np.mean(list(kappa_dict.values()))
kappa_std = np.std(list(kappa_dict.values()))

overall_accuracy = np.mean(list(accuracy_dict.values()))
overall_accuracy_std = np.std(list(accuracy_dict.values()))

print(f'Mean 4-class accuracy: {overall_accuracy*100:.2f}% ± {overall_accuracy_std*100:.2f}%')
print(f'Mean kappa: {final_kappa:.3f} ± {kappa_std:.3f}')
for subject in subjects:
    print(f'subject {subject} - accuracy {accuracy_dict[subject]*100:.2f}% - kappa {kappa_dict[subject]:.3f}')

Mean 4-class accuracy: 63.73% ± 11.93%
Mean kappa: 0.516 ± 0.159
subject A01 - accuracy 74.65% - kappa 0.662
subject A02 - accuracy 49.31% - kappa 0.324
subject A03 - accuracy 77.78% - kappa 0.704
subject A04 - accuracy 57.29% - kappa 0.431
subject A05 - accuracy 50.35% - kappa 0.338
subject A06 - accuracy 47.92% - kappa 0.306
subject A07 - accuracy 79.17% - kappa 0.722
subject A08 - accuracy 68.40% - kappa 0.579
subject A09 - accuracy 68.75% - kappa 0.583
